# Phase 3B — 평가, 통계, control

**목적:** natural primary metrics, task-wise paired differences, calibration, controls, weak-label audit를 검사하고 확대 여부를 결정한다.  
**입력:** Phase 3B result JSON과 per-sample predictions, Phase 3A weak-label audit.  
**출력:** task-first table, paired/clustered-bootstrap analysis, documented gate decision.

In [ ]:
from pathlib import Path
from statistics import mean
import hashlib, json, os, sys, subprocess
root = Path(os.environ.get('GRAPH_CLAD_PROJECT_ROOT', Path.cwd())).resolve()
if not (root / 'scripts').is_dir() and Path('/content/Graph-CLaD').is_dir(): root = Path('/content/Graph-CLaD')
os.chdir(root); sys.path.insert(0, str(root)) if str(root) not in sys.path else None
from scripts.research_paths import resolve_research_paths
paths = resolve_research_paths(project_root=root)
corrected_root = paths.artifact_root / 'phase3_holder_action_v1' / 'corrected_protocol_v2'
threefold_path = corrected_root / 'pair_local_temporal_threefold_seed0_v1' / 'phase3_pair_local_temporal_threefold_seed0_v1.json'
alignment_path = corrected_root / 'pair_local_temporal_action_alignment_seed0_v1' / 'phase3_pair_local_temporal_action_alignment_seed0_v1.json'

## Metric contract
`conditional_oracle_current_change_event`는 current holding을 안다는 조건이다. End-to-end event를 함께 저장한다. Natural test가 primary이고 thresholded F1은 secondary다. Challenge는 stress analysis만 수행한다.

In [ ]:
def natural_metrics(row):
    holding = row['natural_test']['correct']['holding']
    event = holding['conditional_oracle_current_change_event']
    return {
        'pr_auc': event['pr_auc'], 'f1': event['f1'],
        'release_f1': holding['release']['f1'],
        'hard_negative_fpr': holding['hard_negative']['false_positive_rate'],
        'brier': event['brier_score'], 'ece': event['ece'],
        'threshold': row['training']['holding_threshold'],
    }
if not threefold_path.exists(): raise FileNotFoundError(threefold_path)
threefold = json.loads(threefold_path.read_text(encoding='utf-8'))
assert threefold.get('status') == 'completed' and len(threefold.get('results', [])) == 12
task_table = [(r['fold'], r['comparison_id'], natural_metrics(r)) for r in threefold['results']]
task_table[:4]

In [ ]:
# H3 aligned action과 train-shuffled action을 같은 fold/seed로 비교한다.
alignment_summary = {'status': 'missing_or_running'}
if alignment_path.exists():
    shuffled = json.loads(alignment_path.read_text(encoding='utf-8'))
    alignment_summary = {'status': shuffled.get('status'), 'runs': len(shuffled.get('results', []))}
    if shuffled.get('status') == 'completed' and len(shuffled.get('results', [])) == 3:
        aligned = {r['fold']: natural_metrics(r) for r in threefold['results'] if r['comparison_id'] == 'H3-history-action'}
        control = {r['fold']: natural_metrics(r) for r in shuffled['results']}
        differences = {fold: {k: aligned[fold][k] - control[fold][k] for k in ('pr_auc','f1','release_f1','hard_negative_fpr')} for fold in aligned}
        alignment_summary['differences'] = differences
        alignment_summary['positive_pr_auc_tasks'] = sum(v['pr_auc'] > 0 for v in differences.values())
        alignment_summary['macro'] = {k: mean(v[k] for v in differences.values()) for k in next(iter(differences.values()))}
alignment_summary

In [ ]:
# Human weak-label review는 자동 판정으로 대체하지 않는다.
audit_root = corrected_root / 'weak_label_audit_v2_trajectory'
viewer_cmd = [sys.executable, '-m', 'scripts.phase3.weak_label_audit_viewer', '--help']
{'audit_root': str(audit_root), 'exists': audit_root.exists(),
 'required_review': {'tasks': [0,1,2], 'onset_each': 10, 'release_each': 10, 'hard_negative_each': 10},
 'viewer_help_command': ' '.join(viewer_cmd)}

## Gate와 다음 연구 단계
H3 aligned action이 shuffled control보다 natural PR-AUC에서 최소 2개 task 우세하고 release/hard-negative가 안전하며 weak-label review가 충분해야 seeds 1/2 확대를 검토한다. 그 다음 공식 단계는 아직 gated인 Phase 3C CLaD-aligned foresight bridge다. Phase 4 Stage 1 통합이나 Phase 5 이후로 건너뛰지 않는다.